In [ ]:
import os
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin


In [12]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

URL = "https://www.freepik.com/search?format=search&last_filter=query&last_value=Human+hand+-illustration&query=Human+hand+-illustration&selection=1#uuid=5ad5f974-d493-4bbd-bc2e-9e975f1fbb30"

In [13]:
# Test downloading an image
test_image_url = 'https://img.freepik.com/free-psd/reaching-hand-closeup-studio-shot-human-hand_191095-82539.jpg?ga=GA1.1.753861109.1777023105&semt=ais_hybrid&w=740&q=80'

print(f"Testing image download from: {test_image_url}")
print("-" * 60)

try:
    response = requests.get(test_image_url, headers=HEADERS, timeout=10)
    response.raise_for_status()
    
    print(f"✓ Request successful!")
    print(f"  Status Code: {response.status_code}")
    print(f"  Content Type: {response.headers.get('content-type', 'Not specified')}")
    print(f"  Content Length: {len(response.content)} bytes")
    
    # Try to save the image
    test_output_path = os.path.join('scrapped_data', 'test_image.jpg')
    with open(test_output_path, 'wb') as f:
        f.write(response.content)
    
    print(f"✓ Image saved to: {test_output_path}")
    
except requests.RequestException as e:
    print(f"✗ Error downloading image: {e}")
except Exception as e:
    print(f"✗ Unexpected error: {e}")

Testing image download from: https://img.freepik.com/free-psd/reaching-hand-closeup-studio-shot-human-hand_191095-82539.jpg?ga=GA1.1.753861109.1777023105&semt=ais_hybrid&w=740&q=80
------------------------------------------------------------
✓ Request successful!
  Status Code: 200
  Content Type: image/jpeg
  Content Length: 77519 bytes
✓ Image saved to: scrapped_data\test_image.jpg


In [14]:
# Create output directory if it doesn't exist
output_dir = os.path.join('scrapped_data', 'hand_images')
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")
print(f"Directory exists: {os.path.exists(output_dir)}")

Output directory: scrapped_data\hand_images
Directory exists: True


In [35]:
def scrape_images_with_selenium(url, output_dir, max_scroll_pause_time=4, target_images=200):
    """
    Scrape images from Freepik using Selenium and BeautifulSoup.
    Uses keyboard-based scrolling and intelligent Load More button clicking.
    
    Args:
        url: The URL to scrape
        output_dir: Directory to save downloaded images
        max_scroll_pause_time: Time to wait between scrolls for API to respond
        target_images: Continue scrolling if fewer than this many images found
    
    Returns:
        List of downloaded image file paths
    """
    
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    driver = webdriver.Chrome(options=chrome_options)
    downloaded_images = []
    
    try:
        print(f"Opening URL: {url}")
        driver.get(url)
        print("Waiting for page to load...")
        time.sleep(8)
        
        # Click to focus the page
        print("Focusing page...")
        body = driver.find_element(By.TAG_NAME, 'body')
        body.click()
        time.sleep(1)
        
        print("\nInspecting page structure...")
        scrollable_elements = driver.find_elements(By.XPATH, "//*[contains(@style, 'overflow')]")
        print(f"Found {len(scrollable_elements)} scrollable elements")
        
        # Track unique images
        image_urls_found = {}
        consecutive_no_new = 0
        scroll_count = 0
        button_click_count = 0
        
        print("\nStarting scrolling process with Load More button support...\n")
        
        while scroll_count < 200 and consecutive_no_new < 3:
            scroll_count += 1
            
            # Extract image URLs
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            found_new = False
            
            for img in soup.find_all('img'):
                src = img.get('src', '')
                data_src = img.get('data-src', '')
                
                for url_candidate in [src, data_src]:
                    if url_candidate and 'img.freepik.com' in url_candidate and len(url_candidate) > 20:
                        if url_candidate not in image_urls_found:
                            image_urls_found[url_candidate] = True
                            found_new = True
            
            total_unique = len(image_urls_found)
            
            if found_new:
                consecutive_no_new = 0
                print(f"  Scroll #{scroll_count}: Found {total_unique} unique URLs ✓ NEW")
            else:
                consecutive_no_new += 1
                if consecutive_no_new % 3 == 1:
                    print(f"  Scroll #{scroll_count}: {total_unique} URLs [No new: {consecutive_no_new}/3]")
            
            # Stop if target reached
            if total_unique >= target_images:
                print(f"\n✓ Reached target of {target_images} images!")
                break
            
            # Try to click Load More button if scrolling isn't finding new images
            if consecutive_no_new >= 3:
                try:
                    # Find the button with explicit wait
                    load_more_button = WebDriverWait(driver, 3).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="load-more-button"]'))
                    )
                    
                    if load_more_button.is_displayed():
                        print(f"  Found 'Load More' button! Attempting click...")
                        
                        # Scroll to button
                        driver.execute_script("arguments[0].scrollIntoView(true);", load_more_button)
                        time.sleep(1)
                        
                        # Try to click with fallbacks
                        click_successful = False
                        
                        # Try JavaScript click first (most reliable)
                        try:
                            driver.execute_script("arguments[0].click();", load_more_button)
                            print(f"    ✓ JavaScript click successful")
                            click_successful = True
                        except:
                            pass
                        
                        # Try regular click if JS didn't work
                        if not click_successful:
                            try:
                                load_more_button.click()
                                print(f"    ✓ Regular click successful")
                                click_successful = True
                            except:
                                pass
                        
                        # Try ActionChains as last resort
                        if not click_successful:
                            try:
                                ActionChains(driver).move_to_element(load_more_button).click().perform()
                                print(f"    ✓ ActionChains click successful")
                                click_successful = True
                            except:
                                pass
                        
                        if click_successful:
                            button_click_count += 1
                            print(f"    ✓ Button click #{button_click_count} successful")
                            consecutive_no_new = 0
                            print(f"    Waiting 6 seconds for new images...")
                            time.sleep(6)
                            continue
                        else:
                            print(f"    ✗ All click methods failed for this button")
                    else:
                        print(f"  'Load More' button found but not visible")
                except Exception as e:
                    print(f"  'Load More' button not found (end of results): {type(e).__name__}")
            
            # Regular scrolling
            print(f"    Scrolling page...")
            for _ in range(3):
                body.send_keys(Keys.PAGE_DOWN)
                time.sleep(0.5)
            
            if scroll_count % 10 == 0:
                body.send_keys(Keys.END)
                time.sleep(1)
            
            time.sleep(max_scroll_pause_time)
        
        print(f"\n✓ Scrolling complete. Found {len(image_urls_found)} Freepik image URLs")
        
        # Download images
        if len(image_urls_found) == 0:
            print("⚠ No images found!")
            return downloaded_images
        
        print(f"\nDownloading {len(image_urls_found)} images...")
        download_count = 0
        
        for idx, img_url in enumerate(image_urls_found.keys(), 1):
            try:
                if 'placeholder' in img_url.lower() or 'avatar' in img_url.lower():
                    continue
                
                if idx % 50 == 0 or idx <= 2:
                    print(f"  [{idx}/{len(image_urls_found)}] Downloading...")
                
                response = requests.get(img_url, headers=HEADERS, timeout=15)
                response.raise_for_status()
                
                content_type = response.headers.get('content-type', 'image/jpeg').lower()
                ext = '.jpg'
                if 'png' in content_type:
                    ext = '.png'
                elif 'webp' in content_type:
                    ext = '.webp'
                
                filename = f"hand_image_{download_count + 1:04d}{ext}"
                filepath = os.path.join(output_dir, filename)
                
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                
                downloaded_images.append(filepath)
                download_count += 1
                
            except Exception:
                continue
        
        print(f"\n✓ Successfully downloaded {len(downloaded_images)} images!")
        
    except Exception as e:
        print(f"Error during scraping: {e}")
        import traceback
        traceback.print_exc()
    finally:
        driver.quit()
        print("Browser closed")
    
    return downloaded_images

In [36]:
# Run the scraper with improved strategy
print("=" * 80)
print("Starting Web Scraper - Downloading Hand Images")
print("=" * 80)

downloaded_files = scrape_images_with_selenium(
    url=URL,
    output_dir=output_dir,
    max_scroll_pause_time=5,     # Wait 5 seconds for API responses
    target_images=400            # Keep scrolling until we find 200 images
)

# Print summary
print("\n" + "=" * 80)
print("Download Summary")
print("=" * 80)
print(f"Total images downloaded: {len(downloaded_files)}")
print(f"Save location: {output_dir}")
if downloaded_files:
    print("\nFirst few downloaded files:")
    for filepath in downloaded_files[:5]:
        print(f"  - {os.path.basename(filepath)}")

Starting Web Scraper - Downloading Hand Images
Opening URL: https://www.freepik.com/search?format=search&last_filter=query&last_value=Human+hand+-illustration&query=Human+hand+-illustration&selection=1#uuid=5ad5f974-d493-4bbd-bc2e-9e975f1fbb30
Waiting for page to load...
Focusing page...

Inspecting page structure...
Found 3 scrollable elements

Starting scrolling process with Load More button support...

  Scroll #1: Found 23 unique URLs ✓ NEW
    Scrolling page...
  Scroll #2: Found 40 unique URLs ✓ NEW
    Scrolling page...
  Scroll #3: Found 59 unique URLs ✓ NEW
    Scrolling page...
  Scroll #4: Found 75 unique URLs ✓ NEW
    Scrolling page...
  Scroll #5: Found 90 unique URLs ✓ NEW
    Scrolling page...
  Scroll #6: Found 101 unique URLs ✓ NEW
    Scrolling page...
  Scroll #7: Found 121 unique URLs ✓ NEW
    Scrolling page...
  Scroll #8: Found 135 unique URLs ✓ NEW
    Scrolling page...
  Scroll #9: Found 150 unique URLs ✓ NEW
    Scrolling page...
  Scroll #10: Found 169 uniqu